In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Clone your repo into /content
REPO_URL = "https://github.com/ernestgao/Tone_Classifier.git"
PROJECT_DIR = "/content/tone_classifier_project"

%cd /content
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!pwd
!ls -lah

In [ ]:
# Cell 3: Install dependencies
%cd "$PROJECT_DIR"
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install -e .

In [ ]:
# Cell 4: Check GPU
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 5: Auto-download and prepare Stanford politeness data
%cd "$PROJECT_DIR"
!python -m tone_classifier.prepare_data --output_dir data
!ls -lah data

In [ ]:
# Cell 6: Train model
%cd "$PROJECT_DIR"
!python -m tone_classifier.train \
  --train_file "$PROJECT_DIR/data/train.csv" \
  --validation_file "$PROJECT_DIR/data/valid.csv" \
  --test_file "$PROJECT_DIR/data/test.csv" \
  --text_column text \
  --label_column label \
  --model_name microsoft/deberta-v3-base \
  --seed 21 \
  --per_device_train_batch_size 16 \
  --per_device_eval_batch_size 32 \
  --gradient_accumulation_steps 2 \
  --max_length 128 \
  --num_train_epochs 8 \
  --learning_rate 2e-5 \
  --weight_decay 0.01 \
  --warmup_ratio 0.06 \
  --eval_steps 100 \
  --early_stopping_patience 3 \
  --dataloader_num_workers 2 \
  --fp16 \
  --output_dir "$PROJECT_DIR/artifacts/deberta_politeness"

In [ ]:
# Cell 7: Export .pt
%cd "$PROJECT_DIR"
!python -m tone_classifier.export_pt \
  --hf_model_dir "$PROJECT_DIR/artifacts/deberta_politeness/hf_model" \
  --output_pt "$PROJECT_DIR/artifacts/model/politeness_deberta.pt"

In [ ]:
# Cell 8: Quick inference test
%cd "$PROJECT_DIR"
!python -m tone_classifier.predict \
  --hf_model_dir "$PROJECT_DIR/artifacts/deberta_politeness/hf_model" \
  --text "Could you please help me review this when you have time?"

In [ ]:
# Cell 9: Show metrics
import json, os
metrics_path = os.path.join(PROJECT_DIR, "artifacts/deberta_politeness/metrics.json")
with open(metrics_path, "r") as f:
    m = json.load(f)
print("Validation accuracy:", m.get("validation", {}).get("eval_accuracy"))
print("Validation macro_f1:", m.get("validation", {}).get("eval_macro_f1"))
print("Test accuracy:", m.get("test", {}).get("test_accuracy"))
print("Test macro_f1:", m.get("test", {}).get("test_macro_f1"))